In [ ]:
import os as _ka_os2
WORKING_DIR = "/kaggle/working" if _ka_os2.path.exists("/kaggle/working") else _ka_os2.getcwd()
SCRATCH_DIR = "/kaggle/tmp" if _ka_os2.path.exists("/kaggle/tmp") else _ka_os2.getcwd()


In [ ]:
import sys as _ka_sys
try:
    _ka_sys.stdout.reconfigure(line_buffering=True)
    _ka_sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass


In [ ]:
# ==========================================
# AUTO-INJECTED WORKLOAD SHARD CONFIGURATION (Standalone fallback)
# Overridden by the distributed runner at push time
# ==========================================
import os as _ka_os

def _get_shard_var(name, default=None, cast=None):
    val = globals().get(name)
    if val is not None:
        return val
    env_val = _ka_os.environ.get(name)
    if env_val is not None and str(env_val).strip():
        if cast is not None:
            try:
                return cast(env_val)
            except (TypeError, ValueError):
                return default
        if default is not None:
            try:
                return type(default)(env_val)
            except (TypeError, ValueError):
                return default
        return env_val
    return default

SHARD_ID = _get_shard_var('SHARD_ID', 0, cast=int)
TOTAL_SHARDS = _get_shard_var('TOTAL_SHARDS', 1, cast=int)
START_INDEX = _get_shard_var('START_INDEX', None, cast=int)
END_INDEX = _get_shard_var('END_INDEX', None, cast=int)
TOTAL_ITEMS = _get_shard_var('TOTAL_ITEMS', None, cast=int)
SHARD_PARAMS = {}


In [ ]:
%%capture
import os, re
import torch
_tm = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__))
_tv = _tm.group(0) if _tm else "2.8"
xformers = 'xformers==' + {'2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'}.get(_tv, "0.0.34")
!uv pip install --system sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!uv pip install --system --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!uv pip install --system --no-deps --upgrade "torchao>=0.16.0"
!uv pip install --system --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!uv pip install --system torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;


In [ ]:
%%capture
!uv pip install --system --no-deps --upgrade timm # For Gemma 4 vision/audio


In [ ]:
import os
os.makedirs(SCRATCH_DIR, exist_ok=True)
os.makedirs(WORKING_DIR, exist_ok=True)
!wget -O {SCRATCH_DIR}/tamil_transcripts.jsonl https://github.com/barryallen16/sivabharani-comments/raw/refs/heads/main/transcripts/output/tamil_transcripts.jsonl
!wget -O {SCRATCH_DIR}/already_processed_transcripts.jsonl https://github.com/barryallen16/sivabharani-comments/raw/refs/heads/main/intermediate-results/already_processed_transcripts.jsonl


In [ ]:
import hashlib
import json
import os

ALREADY_PROCESSED_TRANSCRIPT_HASHES = set()
_RESUME_FILES = [
    os.path.join(SCRATCH_DIR, "already_processed_transcripts.jsonl"),
    os.path.join(WORKING_DIR, f"already_processed_transcripts_shard{SHARD_ID}.jsonl"),
]
for _rf in _RESUME_FILES:
    try:
        with open(_rf, "r", encoding="utf-8") as in_file:
            for line in in_file:
                if line.strip():
                    data = json.loads(line)
                    video_id = data["video_id"]
                    text = data["text"]
                    transcript_hash = hashlib.sha256(f"{video_id} + {text}".encode()).hexdigest()
                    if transcript_hash not in ALREADY_PROCESSED_TRANSCRIPT_HASHES:
                        ALREADY_PROCESSED_TRANSCRIPT_HASHES.add(transcript_hash)
    except FileNotFoundError:
        continue
print(f"resume skip-set: {len(ALREADY_PROCESSED_TRANSCRIPT_HASHES)} hashes from {[f for f in _RESUME_FILES]}", flush=True)


In [ ]:
from unsloth import FastModel
import torch
import sys
if torch.cuda.device_count() == 0:
    print("ERROR: no GPU visible; this job needs an accelerator (T4-x2).", flush=True)
    sys.exit(1)

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] 
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    dtype = None, # None for auto detection
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)


In [ ]:
def gemma_do_inference(messages, max_new_tokens):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=1.0,  top_p = 0.95, top_k = 64)
    prompt_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][prompt_length:]
    final_result = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return final_result

In [ ]:
def process_transcript_chunks(INPUT):
    
    # INPUT = "சரி ஓகே. என்ஜிகே அப்படிங்கிற நண்பர் வந்து ப்ரோ ஹாய் ப்ரோ நியூ அப்டேட்ல ஏர்பாட் சப்போர்ட் அண்ட் சம் அதர் பீச்சர்ஸ்ும் வந்திருக்கு. நான் F8"
    PROMPT="""
    ### System Instruction:
    You are a linguistic data extraction tool. Your goal is to identify human names in Tamil text and provide a JSON response containing the original Tamil names and their English transliterations.
    
    ### Constraints:
    - Only extract names of people.
    - Do not include words like "Bye Bye", "Hello", or "Location names".
    - Ensure the English list matches the order of the Tamil list.
    - If no names exist, return: {"tamil": [], "english": []}
    
    ### Examples:
    Input: வணக்கம், நான் சூர்யா.
    Output: {"tamil": ["சூர்யா"], "english": ["Surya"]}
    
    Input: இன்று ரவி மற்றும் கவி வந்தார்கள்.
    Output: {"tamil": ["ரவி", "கவி"], "english": ["Ravi", "Kavi"]}
    
    Input: நான் இப்போது செல்கிறேன். பாய் பாய்.
    Output: {"tamil": [], "english": []}
    
    ### Task:
    Extract names from the following input text:
    """
    PROMPT += f"""
    Input: {INPUT}
    JSON Output:
    """
    messages = [{
        "role" : "user",
        "content": [
            { "type": "text",  "text" : PROMPT }
        ]
    }]
    # You might have to wait 1 minute for Unsloth's auto compiler
    results = gemma_do_inference(messages, max_new_tokens = 256)
    cleaned_output = results.replace("json", "").replace("```", "").strip()
    return cleaned_output

In [ ]:
from tqdm import tqdm
import os

INPUT_PATH = os.path.join(SCRATCH_DIR, "tamil_transcripts.jsonl")
OUTPUT_PATH = os.path.join(WORKING_DIR, f"already_processed_transcripts_shard{SHARD_ID}.jsonl")

def _chunk_text(chunk):
    return chunk.get("text", chunk.get(" text", ""))

# Preserve input order; drop videos already fully covered by the resume skip-set
# so shard slices split REMAINING work, not total work.
_video_items = []
with open(INPUT_PATH, "r", encoding="utf-8") as in_file:
    for line in in_file:
        if line.strip():
            data = json.loads(line)
            for video_id, chunks in data.items():
                _video_items.append((video_id, chunks))

_remaining = []
for _vid, _ch in _video_items:
    _keep = False
    for _c in _ch:
        _h = hashlib.sha256(f"{_vid} + {_chunk_text(_c)}".encode()).hexdigest()
        if _h not in ALREADY_PROCESSED_TRANSCRIPT_HASHES:
            _keep = True
            break
    if _keep:
        _remaining.append((_vid, _ch))
print(f"videos: total={len(_video_items)} remaining={len(_remaining)} shard={SHARD_ID}/{TOTAL_SHARDS} slice=[{START_INDEX}:{END_INDEX}]", flush=True)

if START_INDEX is not None and END_INDEX is not None:
    _work = _remaining[START_INDEX:END_INDEX]
else:
    _work = _remaining[SHARD_ID::TOTAL_SHARDS]
print(f"this shard: {len(_work)} videos", flush=True)

for video_id, chunks in tqdm(_work, desc="Processing video transcripts", unit="video"):
    for chunk in chunks:
        input_text = _chunk_text(chunk)
        transcript_hash = hashlib.sha256(f"{video_id} + {input_text}".encode()).hexdigest()
        if transcript_hash in ALREADY_PROCESSED_TRANSCRIPT_HASHES:
            continue
        start = chunk.get("start")
        duration = chunk.get("duration")
        result = process_transcript_chunks(input_text)
        if result:
            try:
                parsed_extract = json.loads(result)
                result_dict = {
                    "video_id": video_id,
                    "text": input_text,
                    "extract": parsed_extract,
                    "start": start,
                    "duration": duration
                }
                print(result_dict, flush=True)
                with open(OUTPUT_PATH, "a", encoding="utf-8") as out_file:
                    json.dump(result_dict, out_file, ensure_ascii=False)
                    out_file.write("\n")
                ALREADY_PROCESSED_TRANSCRIPT_HASHES.add(transcript_hash)
            except json.JSONDecodeError:
                continue


In [ ]:
from huggingface_hub import HfApi, create_repo
import os

LOCAL_FILE_NAME = os.path.join(WORKING_DIR, f"already_processed_transcripts_shard{SHARD_ID}.jsonl")
FILE_PATH_IN_REPO = f"data/processed_transcripts_shard{SHARD_ID}.jsonl"
REPO_ID = "barryallen16/sivabharani_comments"
# Token comes from the kaggle-automation env preamble at push time; fall back to
# Kaggle secrets only for standalone runs.
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("WRITE_HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("WRITE_HF_TOKEN")
    except Exception as _e:
        print(f"no HF token available (env + secrets): {_e}", flush=True)
        HF_TOKEN = None
try:
    if not HF_TOKEN:
        raise RuntimeError("HF token missing: set it in the kaggle-automation env.")
    api = HfApi(token=HF_TOKEN)
    create_repo(repo_id=REPO_ID,
                repo_type="dataset",
                exist_ok=True,
                token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=LOCAL_FILE_NAME,
        path_in_repo=FILE_PATH_IN_REPO,
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message=f"Shard {SHARD_ID} processing output.",
    )
    print(f"pushed {LOCAL_FILE_NAME} to {REPO_ID}:{FILE_PATH_IN_REPO}", flush=True)
except Exception as e:
    print(f"error occured: {e}", flush=True)
